In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

os.makedirs("../results/figures", exist_ok=True)
pd.set_option("display.float_format", "{:.3f}".format)

print("Librairies chargées ✓")

Librairies chargées ✓


In [8]:
# Chargement des résultats du backtesting
results = pd.read_csv(
    "../results/backtesting_results.csv",
    parse_dates=["date"]
)

print(f"Période analysée : {results['date'].min().date()}"
      f" → {results['date'].max().date()}")
print(f"Nombre de mois   : {len(results)}")
print(f"\nAperçu :")
print(results[[
    "month", "n_selected", "ml_return", "bh_return",
    "ml_cumulative", "bh_cumulative"
]])

Période analysée : 2022-01-03 → 2022-12-01
Nombre de mois   : 12

Aperçu :
      month  n_selected  ml_return  bh_return  ml_cumulative  bh_cumulative
0   2022-01          10      0.054      0.035     105380.982     103515.809
1   2022-02           9      0.138      0.054     119888.437     109115.157
2   2022-03          10      0.014      0.022     121573.735     111567.993
3   2022-04           8     -0.110     -0.106     108153.789      99780.550
4   2022-05          10     -0.002     -0.016     107954.644      98141.036
5   2022-06          11     -0.125     -0.070      94466.728      91254.278
6   2022-07           8      0.032      0.030      97474.334      94001.698
7   2022-08           9     -0.034     -0.009      94175.628      93124.319
8   2022-09           3      0.046     -0.029      98470.665      90421.386
9   2022-10           4     -0.015      0.024      96968.952      92556.973
10  2022-11           5      0.108      0.078     107425.810      99771.155
11  2022-12  

## Backtesting de la stratégie ML

Le backtesting simule ce qui se serait passé si on avait suivi les recommandations du modèle en conditions réelles.

On compare deux stratégies sur l'année 2022 :

- **Stratégie ML** : on investit uniquement dans les actions que le modèle recommande avec une confiance ≥ 55%
- **Stratégie Buy & Hold** : on investit dans toutes les actions sans sélection — c'est notre référence

Capital de départ : **100 000 ZAR**

In [9]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=results["date"],
    y=results["ml_cumulative"],
    mode="lines+markers",
    name="Stratégie ML",
    line=dict(color="#1D9E75", width=2.5),
    marker=dict(size=7)
))

fig.add_trace(go.Scatter(
    x=results["date"],
    y=results["bh_cumulative"],
    mode="lines+markers",
    name="Buy & Hold",
    line=dict(color="#D85A30", width=2.5),
    marker=dict(size=7)
))

fig.add_hline(
    y=100_000,
    line_dash="dash",
    line_color="gray",
    annotation_text="Capital initial : 100 000 ZAR"
)

fig.update_layout(
    title="Performance cumulée — ML vs Buy & Hold (2022)",
    xaxis_title="Date",
    yaxis_title="Valeur du portefeuille (ZAR)",
    template="plotly_white",
    width=800, height=500,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

fig.write_html("../results/figures/04_performance_cumulee.html")
fig.write_image("../results/figures/04_performance_cumulee.png")

Resorting to unclean kill browser.


RuntimeError: Couldn't close or kill browser subprocess

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=results["month"],
    y=results["ml_return"] * 100,
    name="Stratégie ML",
    marker_color="#1D9E75",
    opacity=0.8
))

fig.add_trace(go.Bar(
    x=results["month"],
    y=results["bh_return"] * 100,
    name="Buy & Hold",
    marker_color="#D85A30",
    opacity=0.8
))

fig.add_hline(
    y=0, line_color="gray", line_width=1
)

fig.update_layout(
    title="Rendements mensuels — ML vs Buy & Hold (2022)",
    xaxis_title="Mois",
    yaxis_title="Rendement (%)",
    template="plotly_white",
    barmode="group",
    width=800, height=450,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

fig.write_html("../results/figures/04_rendements_mensuels.html")
fig.write_image("../results/figures/04_rendements_mensuels.png")

In [ ]:
fig = px.bar(
    results,
    x="month",
    y="n_selected",
    title="Nombre d'actions sélectionnées par le modèle chaque mois",
    labels={
        "month"      : "Mois",
        "n_selected" : "Nombre d'actions"
    },
    color="n_selected",
    color_continuous_scale=["#F5C4B3", "#1D9E75"],
    template="plotly_white",
    width=750, height=400
)

fig.update_layout(coloraxis_showscale=False)
fig.show()

fig.write_html("../results/figures/04_actions_selectionnees.html")
fig.write_image("../results/figures/04_actions_selectionnees.png")

In [ ]:
# Résumé des métriques clés
ml_final  = results["ml_cumulative"].iloc[-1]
bh_final  = results["bh_cumulative"].iloc[-1]
capital   = 100_000

summary = pd.DataFrame({
    "Métrique": [
        "Capital initial (ZAR)",
        "Capital final (ZAR)",
        "Rendement total",
        "Rendement mensuel moyen",
        "Meilleur mois",
        "Pire mois",
    ],
    "Stratégie ML": [
        f"{capital:,.0f}",
        f"{ml_final:,.0f}",
        f"{(ml_final - capital) / capital:.1%}",
        f"{results['ml_return'].mean():.1%}",
        f"{results['ml_return'].max():.1%}",
        f"{results['ml_return'].min():.1%}",
    ],
    "Buy & Hold": [
        f"{capital:,.0f}",
        f"{bh_final:,.0f}",
        f"{(bh_final - capital) / capital:.1%}",
        f"{results['bh_return'].mean():.1%}",
        f"{results['bh_return'].max():.1%}",
        f"{results['bh_return'].min():.1%}",
    ]
})

summary.set_index("Métrique", inplace=True)
summary

,Stratégie ML,Buy & Hold
Métrique,,
Capital initial (ZAR),"100,000","100,000"
Capital final (ZAR),"106,496","101,277"
Rendement total,6.5%,1.3%
Rendement mensuel moyen,0.8%,0.2%
Meilleur mois,13.8%,7.8%
Pire mois,-12.5%,-10.6%


##  Limites importantes du backtesting

Tout backtesting doit être lu avec prudence. Voici les principales limites à garder en tête.

**Biais de survie** — on n'a gardé que les entreprises encore cotées en 2024. Les entreprises qui ont fait faillite ou ont été retirées de la cote ne sont pas dans notre dataset, ce qui surestime légèrement les rendements.

**Coûts de transaction** — on n'a pas pris en compte les frais de courtage, la fiscalité sur les plus-values, ni le spread bid-ask. En pratique ces coûts réduisent le rendement réel.

**Données statiques** — comme mentionné précédemment, les fondamentaux ne changent pas dans notre modèle. Un vrai système de trading mettrait à jour ces données chaque trimestre.

**Une seule année de test** — 2022 est une seule année. Des résultats robustes nécessiteraient un test sur plusieurs cycles de marché.

Malgré ces limites, le résultat de **+5.2% d'avantage sur le Buy & Hold** est encourageant et montre que la sélection ML apporte de la valeur.